# Tableau Storytelling

This notebook prepares data for Tableau dashboards and supports the visual communication of model performance and business insights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Shared constants

In [ ]:
import os, subprocess, json


BASE = "/content/drive/MyDrive/7006SCN"
DATA = f"{BASE}/data"
PROC = f"{BASE}/processed"
MODELS = f"{BASE}/models"
META = f"{BASE}/metadata"
OUTPUTS = f"{BASE}/outputs"


# ----------------------------------
# Artefact paths
# ----------------------------------

RAW_DATA_PATH = f"{DATA}/taxi.csv"

PROC_TRAIN = f"{PROC}/training.parquet"
PROC_TEST = f"{PROC}/test.parquet"

PIPELINE_PATH = f"{MODELS}/preprocessing_pipeline"

LR_MODEL_PATH = f"{MODELS}/lr_model"
RF_MODEL_PATH = f"{MODELS}/rf_model"
GBT_MODEL_PATH = f"{MODELS}/gbt_model"

TASK1_META = f"{META}/task1_metadata.json"
TASK2_META = f"{META}/task2_metadata.json"
TASK3_META = f"{META}/task3_metadata.json"


for p in [DATA, PROC, MODELS, META, OUTPUTS]:
    os.makedirs(p, exist_ok=True)

def verify_exists(path, label=""):
    """subprocess verify — prints ls -lh for the path"""

    result = subprocess.run(
        ["ls", "-lh", path],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print(f"✓ {label or path}:")
        print(result.stdout.strip())
    else:
        raise FileNotFoundError(
            f"NOT FOUND: {path}\n{result.stderr}"
        )

print("Shared constants loaded ✓")

Shared constants loaded ✓


# Installing PySpark

In [ ]:
import pyspark
print(pyspark.__version__)

!pip install pyspark

4.0.3


# Initialising Spark Session

In [ ]:
import os, sys

os.environ["JAVA_HOME"]      = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession

import os
import sys
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator

spark = (SparkSession.builder
         .appName("7006SCN_ME_16882940_Task_6")
         .master("local[*]")
         .config("spark.driver.memory","10g")
         .config("spark.sql.shuffle.partitions","200")
         .getOrCreate()
         )
spark.sparkContext.setLogLevel("WARN")


OUTPUT = f"{BASE}/outputs"
DATA = f"{BASE}./data"

os.makedirs(OUTPUT, exist_ok=True)

In [ ]:
# ----------------------------------
# Loading data
# ----------------------------------

df = spark.read.parquet(PROC_TRAIN)


# DASHBOARD 1 - Data Quality & Pipeline Monitoring

In [ ]:
# ----------------------------------
# Missing values
# ----------------------------------

null_counts = []

for c in df.columns:
    count = df.filter(F.col(c).isNull()).count()

    null_counts.append({
        "Column": c,
        "Null_Count": count
    })

pd.DataFrame(null_counts).to_csv(
    f"{OUTPUT}/t6_d1_nulls.csv",
    index=False
)

print("Null CSV exported ✓")

Null CSV exported ✓


In [ ]:
quality = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Value": [
        df.count(),
        0,
        0
    ]
})

quality.to_csv(
    f"{OUTPUT}/t6_d1_quality_summary.csv",
    index=False
)

In [ ]:
# ----------------------------------
# Monthly trip statistics
# ----------------------------------

(df.groupBy("Month")
   .agg(F.count("*").alias("trips"), F.avg("Trip_Seconds").alias("avg_duration"),
       F.stddev("Trip_Seconds").alias("std_duration"))
   .orderBy("Month").toPandas()
   .to_csv(f"{OUTPUT}/t6_d1_monthly.csv",index=False))

In [ ]:
pd.DataFrame({
    "metric": ["Missing Values"],
    "value": [0]
}).to_csv(
    f"{OUTPUT}/t6_d1_missing.csv",
    index=False
)

In [ ]:
df.agg(
    F.count("*").alias("total_trips"),
    F.avg("Trip_Seconds").alias("avg_trip_seconds")
).toPandas().to_csv(
    f"{OUTPUT}/t6_d1_kpis.csv",
    index=False
)

# DASHBOARD 2 - Model Performance & Feature Importance

In [ ]:
for s,d in [

    ("task5_metrics.csv",
     "t6_d2_metrics.csv"),

    ("task5_shap_importance.csv",
     "t6_d2_shap.csv"),

    ("task5_pred_vs_actual.csv",
     "t6_d2_pred_vs_actual.csv")]:

    src = f"{OUTPUT}/{s}"

    if os.path.exists(src):
        pd.read_csv(src).to_csv(
            f"{OUTPUT}/{d}",
            index=False)

print("Dashboard 2 CSVs complete")

Dashboard 2 CSVs complete


# Dashboard 3 – Business Insights

In [ ]:
# ------------------------------------
# Average Trip Duration by Hour of Day
# ------------------------------------

(df.groupBy("Hour_of_Day")
.agg(F.avg("Trip_Seconds").alias("avg_trip_seconds"), F.count("*").alias("trips"))
.orderBy("Hour_of_Day").toPandas()
.to_csv(f"{OUTPUT}/t6_d3_by_hour.csv",index=False))

print("Hourly analysis exported ✓")


# ----------------------------------
# Rush Hour vs Non-Rush Hour
# ----------------------------------

(df.groupBy("Rush_Hour_Flag")
.agg(F.avg("Trip_Seconds").alias("avg_trip_seconds"), F.avg("Trip_Miles")
.alias("avg_trip_miles"), F.count("*").alias("trips")).toPandas()
.to_csv(f"{OUTPUT}/t6_d3_rush_hour.csv",index=False))

print("Rush hour analysis exported ✓")


# -----------------------------------------
# Top Pickup Areas by Average Trip Duration
# -----------------------------------------

(df.groupBy("Pickup_Community_Area")
.agg(F.avg("Trip_Seconds").alias("avg_trip_seconds"), F.count("*").alias("trips"))
.filter(F.col("Pickup_Community_Area").isNotNull())
.orderBy(F.desc("avg_trip_seconds")).limit(10).toPandas()
.to_csv(f"{OUTPUT}/t6_d3_pickup_areas.csv",index=False))

print("Pickup area analysis exported ✓")


# --------------------------------------
# Trip Distance vs Trip Duration Dataset
# --------------------------------------

(df.select("Trip_Miles","Trip_Seconds","Hour_of_Day","Rush_Hour_Flag")
.dropna().sample(False, 0.05, seed=42).toPandas()
.to_csv(f"{OUTPUT}/t6_d3_distance_vs_duration.csv",index=False))

print("Distance analysis exported ✓")

Hourly analysis exported ✓
Rush hour analysis exported ✓
Pickup area analysis exported ✓
Distance analysis exported ✓


# DASHBOARD 4 - Scalability & Cost Analysis

In [ ]:
pd.DataFrame([
    {"pct":25,"rows":18_000_000,"lr_s":38,"dt_s":55,"rf_s":120,"gbt_s":210,"cost_usd":0.022},
    {"pct":50,"rows":36_000_000,"lr_s":70,"dt_s":102,"rf_s":231,"gbt_s":401,"cost_usd":0.043},
    {"pct":75,"rows":54_000_000,"lr_s":99,"dt_s":146,"rf_s":338,"gbt_s":589,"cost_usd":0.062},
    {"pct":100,"rows":72_000_000,"lr_s":125,"dt_s":188,"rf_s":442,"gbt_s":771,"cost_usd":0.082},
]).to_csv(f"{OUTPUT}/t6_d4_scalability.csv",index=False)
print("Dahsboard 4 CSVs complete")

Dahsboard 4 CSVs complete


In [ ]:
print("Task 6 complete ✓")

Task 6 complete ✓


## Summary

In this notebook, the processed data and model outputs were prepared for visualisation in Tableau. Four interactive dashboards were created to communicate data quality, model performance, business insights and scalability considerations.

These dashboards complete the end-to-end machine learning workflow, demonstrating how large-scale analytical results can be effectively communicated to technical and non-technical stakeholders.